# Citi Bike (New York) → TTE format, **OD mode**

**Read this first.** Citi Bike is *not* route-aware. Every record is
`(start station, end station, start time, end time)` — the duration is **measured**,
the route is **never observed**. The files this notebook writes are in the gold
format so the same loader reads them, but each trip has exactly **two points**:
where it started and where it ended. Anything that consumes `OSMids` as a *path*
will silently treat these as one-hop routes, which they are not. Keep this
dataset in a separate OD-mode split, labelled as such.

**What it is worth anyway:** ~30.8 GB across 173 objects on
`s3.amazonaws.com/tripdata/`, **2013 → July 2026** (verified by listing the
bucket), tens of millions of trips a year with real coordinates in the record,
covering COVID and thirteen years of drift. The label mixes route choice, rider
fitness, e-bike vs pedal, and detours, so the irreducible variance is enormous —
which makes it a hard, honest testbed for **calibration** rather than point
accuracy. Sister buckets, all verified live and listable:

| system | bucket | span |
|---|---|---|
| **Citi Bike** (NYC) | `s3.amazonaws.com/tripdata/` | 2013 → 2026-07, 30.8 GB |
| **Capital Bikeshare** (DC) | `s3.amazonaws.com/capitalbikeshare-data/` | **2010** → 2026-07, 1.7 GB |
| **Divvy** (Chicago) | `divvy-tripdata.s3.amazonaws.com` | 2013 → 2026-07, 1.9 GB |
| **Bay Wheels** (SF) | `s3.amazonaws.com/baywheels-data/` | 2017 → 2026-07, 1.0 GB |

**Licence.** Divvy and Capital Bikeshare forbid redistributing the data as a
stand-alone dataset (CaBi adds a non-commercial clause). Citi Bike's data policy
is permissive for analysis but publish the **loader**, not the files. TfL
Santander Cycles (OGL v2) is the only one of the big systems where
redistribution is clearly fine.

Operators already filtered the source: staff trips, test stations and **every ride
under 60 seconds** are removed, so the left tail of the label distribution is cut.

## Target format (the "gold" contract)

Taken from the Harbin files in `datasets.zip`, with the Omsk file naming:

| file | contents |
|---|---|
| `matched_trips_<city>.csv` | unnamed index, `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time` |
| `edge_list_directed_<city>.csv` | `osmid_u`, `osmid_v` — directed transitions between road segments |
| `road_network_unique_osmids_<city>.geojson` | one `LineString` per segment, `osmid` / `original_osmid` / `is_duplicate` / `duplicate_index` / `length` / `highway` / ... |

`Coordinates`, `OSMids` and `Timestamps` are Python-literal lists of **equal
length — one entry per GPS fix**: `(lon, lat)` floats, the segment id the fix was
matched to (a string), and the unix timestamp in seconds.
`Total_time = Timestamps[-1] - Timestamps[0]`, in seconds.

In [ ]:
CITY    = "nyc_citibike"
OUT     = "."
GRAPHML = "nyc_drive.graphml"
CACHE   = "citibike_raw"

BUCKET = "https://s3.amazonaws.com/tripdata"
MONTHS = ["202506-citibike-tripdata.zip"]     # add more; list the bucket to see names
BBOX   = (-74.05, 40.63, -73.88, 40.85)       # Manhattan + inner Brooklyn/Queens

MIN_SECONDS = 120
MAX_SECONDS = 7200
MIN_METERS  = 300          # straight-line, not path
MAX_SNAP_M  = 120          # stations sit on the kerb, not on the carriageway
MAX_TRIPS   = 200000
RANDOM_SEED = 0

In [ ]:
# pip install pandas numpy osmnx geopandas shapely networkx tqdm requests
import ast, glob, io, json, math, os, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import requests
from shapely.geometry import Point, mapping
from tqdm.auto import tqdm

## Helpers

In [ ]:
# --- unique OSM ids -----------------------------------------------------------
def assign_unique_osmids(G):
    """(u, v, key) -> {unique_osmid, original_osmid, is_duplicate, duplicate_index}.

    One OSM way is split into many graph edges, so `osmid` is not unique.  The
    gold files use `<way_id>` when a way appears once and `<way_id>_<i>`
    (i = 1, 2, ...) for every edge of a way that appears several times -- see
    `Harbin_edge_list.csv` ('858780553' next to '705148575_1').
    """
    base = {}
    for u, v, k, d in G.edges(keys=True, data=True):
        o = d.get("osmid")
        if isinstance(o, (list, tuple, set)):
            o = sorted(o)[0]
        base[(u, v, k)] = str(o)

    counts = pd.Series(list(base.values())).value_counts().to_dict()
    seen, out = {}, {}
    for e, b in base.items():
        if counts[b] == 1:
            out[e] = dict(unique_osmid=b, original_osmid=b,
                          is_duplicate=False, duplicate_index=0)
        else:
            i = seen.get(b, 0) + 1
            seen[b] = i
            out[e] = dict(unique_osmid=f"{b}_{i}", original_osmid=b,
                          is_duplicate=True, duplicate_index=i)
    return out


# --- edge_list_directed_<city>.csv --------------------------------------------
def build_edge_list_directed(G, uid):
    """Directed transitions between segments that share a node (Harbin style)."""
    inc, out = {}, {}
    for u, v, k in G.edges(keys=True):
        out.setdefault(u, []).append((u, v, k))
        inc.setdefault(v, []).append((u, v, k))
    rows = set()
    for node in set(inc) & set(out):
        for e1 in inc[node]:
            for e2 in out[node]:
                if e1 == e2:
                    continue
                a, b = uid[e1]["unique_osmid"], uid[e2]["unique_osmid"]
                if a != b:
                    rows.add((a, b))
    return pd.DataFrame(sorted(rows), columns=["osmid_u", "osmid_v"])


# --- road_network_unique_osmids_<city>.geojson --------------------------------
_KEEP = ["bridge", "highway", "lanes", "name", "oneway", "reversed",
         "junction", "ref", "tunnel", "maxspeed", "access", "width"]


def _clean(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    if isinstance(val, (list, tuple)):
        val = [str(x) for x in val]
        return val or None
    return val if isinstance(val, bool) else str(val)


def build_road_network_geojson(G, uid, path):
    """One LineString per graph edge with the Harbin property set."""
    Gu = G if str(G.graph.get("crs", "")).lower() in ("epsg:4326", "wgs84") \
        else ox.projection.project_graph(G, to_latlong=True)
    edges = ox.convert.graph_to_gdfs(Gu, nodes=False, edges=True, fill_edge_geometry=True)
    feats = []
    for (u, v, k), row in edges.iterrows():
        info = uid[(u, v, k)]
        props = {"u": int(u), "v": int(v), "key": int(k),
                 "osmid": info["original_osmid"],
                 "unique_osmid": info["unique_osmid"]}
        for c in _KEEP:
            props[c] = _clean(row[c]) if c in edges.columns else None
        props["length"] = float(row["length"])
        props["original_osmid"] = info["original_osmid"]
        props["is_duplicate"] = info["is_duplicate"]
        props["duplicate_index"] = info["duplicate_index"]
        props = {a: b for a, b in props.items() if b is not None}
        feats.append({"type": "Feature", "properties": props,
                      "geometry": mapping(row["geometry"])})
    fc = {"type": "FeatureCollection", "name": "road_network_unique_osmids",
          "crs": {"type": "name",
                  "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
          "features": feats}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False)
    return len(feats)


# --- map matching -------------------------------------------------------------
class Matcher:
    """Light HMM map matcher.  Emission = snap distance, transition = adjacency.

    Restricting transitions to {same edge, 1-hop, 2-hop} is enough at 5-60 s
    sampling and, unlike plain nearest-edge snapping, it resolves the direction
    of travel on two-way streets: the reverse edge is simply unreachable.
    """

    def __init__(self, G, radius_m=60.0, sigma_m=20.0, k_candidates=8,
                 p_adjacent=1.0, p_two_hop=4.0, p_break=25.0):
        projected = str(G.graph.get("crs", "")).lower() not in ("epsg:4326", "wgs84", "")
        self.Gp = G if projected else ox.projection.project_graph(G)
        self.crs = self.Gp.graph["crs"]
        self.edges = ox.convert.graph_to_gdfs(self.Gp, nodes=False, edges=True,
                                              fill_edge_geometry=True)
        self.index = list(self.edges.index)
        self.sindex = self.edges.sindex
        self.radius, self.sigma, self.k = radius_m, sigma_m, k_candidates
        self.p_adj, self.p_two, self.p_break = p_adjacent, p_two_hop, p_break
        self.succ = {}
        for u, v, k in self.Gp.edges(keys=True):
            self.succ.setdefault(u, []).append((u, v, k))
        self._two_hop = {}

    def _out(self, e):
        return self.succ.get(e[1], [])

    def _two(self, e):
        if e not in self._two_hop:
            s = set()
            for nxt in self._out(e):
                s.update(self._out(nxt))
            self._two_hop[e] = s
        return self._two_hop[e]

    def _trans(self, e1, e2):
        if e1 == e2:
            return 0.0
        if e2 in self._out(e1):
            return self.p_adj
        if e2 in self._two(e1):
            return self.p_two
        return self.p_break

    def _candidates(self, xs, ys):
        out = []
        for x, y in zip(xs, ys):
            p = Point(x, y)
            hits = self.sindex.query(p.buffer(self.radius), predicate="intersects")
            if len(hits) == 0:
                out.append([])
                continue
            cand = [(self.index[i], self.edges.geometry.iloc[i].distance(p))
                    for i in np.atleast_1d(hits)]
            cand.sort(key=lambda t: t[1])
            out.append(cand[: self.k])
        return out

    def match(self, lon, lat):
        """Viterbi decode one trip.  Returns (edge_per_point, snap_distance_m)."""
        pts = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs(self.crs)
        cands = self._candidates(pts.x.values, pts.y.values)
        if any(len(c) == 0 for c in cands):
            return None, None

        score = {e: 0.5 * (d / self.sigma) ** 2 for e, d in cands[0]}
        back = [{}]
        for t in range(1, len(cands)):
            new, bp = {}, {}
            for e2, d2 in cands[t]:
                best_e, best_s = None, np.inf
                for e1, s1 in score.items():
                    s = s1 + self._trans(e1, e2)
                    if s < best_s:
                        best_s, best_e = s, e1
                new[e2] = best_s + 0.5 * (d2 / self.sigma) ** 2
                bp[e2] = best_e
            score = new
            back.append(bp)

        e = min(score, key=score.get)
        path = [e]
        for t in range(len(cands) - 1, 0, -1):
            e = back[t][e]
            path.append(e)
        path.reverse()
        dist = [dict(c).get(e, np.nan) for c, e in zip(cands, path)]
        return path, np.asarray(dist)


def connectivity(seq):
    """Share of consecutive edge changes where the two edges really share a node."""
    pairs = [(a, b) for a, b in zip(seq, seq[1:]) if a != b]
    if not pairs:
        return 1.0
    return sum(1 for (_, v1, _), (u2, _, _) in pairs if v1 == u2) / len(pairs)


# --- geometry -----------------------------------------------------------------
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def to_epoch(series):
    """Resolution-safe datetime -> unix seconds (pandas 2 and 3).

    NaT survives as NaN rather than raising: callers parse with
    `errors="coerce"` on purpose and drop the bad rows immediately after, so
    casting to int64 here would blow up before they get the chance.
    """
    tz = getattr(series.dtype, "tz", None)
    zero = pd.Timestamp("1970-01-01", tz=tz) if tz is not None else pd.Timestamp("1970-01-01")
    sec = (series - zero) // pd.Timedelta("1s")
    return sec.astype("int64") if sec.notna().all() else sec.astype("float64")

## 1. Fetch the months

The bucket is a plain public S3 listing — `curl https://s3.amazonaws.com/tripdata/`
returns the XML index if you want to see what is available.

In [ ]:
os.makedirs(CACHE, exist_ok=True)
frames = []
for name in MONTHS:
    local = os.path.join(CACHE, name)
    if not os.path.exists(local):
        r = requests.get(f"{BUCKET}/{name}", timeout=600)
        r.raise_for_status()
        with open(local, "wb") as f:
            f.write(r.content)
        print("downloaded", name, round(len(r.content) / 1e6, 1), "MB")
    with zipfile.ZipFile(local) as z:
        for member in z.namelist():
            if member.endswith(".csv") and "__MACOSX" not in member:
                with z.open(member) as fh:
                    frames.append(pd.read_csv(fh, low_memory=False))
raw = pd.concat(frames, ignore_index=True)
del frames
print(raw.shape)
print(list(raw.columns))

In [ ]:
# two schemas exist: the 2013-2020 one and the 2020+ one
LEGACY = {"starttime": "started_at", "stoptime": "ended_at",
          "start station latitude": "start_lat", "start station longitude": "start_lng",
          "end station latitude": "end_lat", "end station longitude": "end_lng",
          "start station id": "start_station_id", "end station id": "end_station_id",
          "start station name": "start_station_name", "end station name": "end_station_name"}
df = raw.rename(columns={k: v for k, v in LEGACY.items() if k in raw.columns})
if "ride_id" not in df.columns:
    df["ride_id"] = np.arange(len(df)).astype(str)
if "rideable_type" not in df.columns:
    df["rideable_type"] = "classic_bike"

df = df.dropna(subset=["started_at", "ended_at", "start_lat", "start_lng", "end_lat", "end_lng"])
df["t0"] = to_epoch(pd.to_datetime(df["started_at"], format="mixed"))
df["t1"] = to_epoch(pd.to_datetime(df["ended_at"], format="mixed"))
df["duration"] = df["t1"] - df["t0"]
df["meters"] = haversine_m(df["start_lng"], df["start_lat"], df["end_lng"], df["end_lat"])
print(len(df), "rides |", pd.to_datetime(df.t0.min(), unit="s"), "->",
      pd.to_datetime(df.t0.max(), unit="s"))
df[["duration", "meters"]].describe()

In [ ]:
inside = (df.start_lng.between(BBOX[0], BBOX[2]) & df.start_lat.between(BBOX[1], BBOX[3])
          & df.end_lng.between(BBOX[0], BBOX[2]) & df.end_lat.between(BBOX[1], BBOX[3]))
sel = df[inside
         & df.duration.between(MIN_SECONDS, MAX_SECONDS)
         & (df.meters >= MIN_METERS)].copy()
print(f"{len(sel)} / {len(df)} rides pass the filters")
if MAX_TRIPS and len(sel) > MAX_TRIPS:
    sel = sel.sample(MAX_TRIPS, random_state=RANDOM_SEED)
print("round trips (same station) dropped by MIN_METERS:",
      int((df.meters < MIN_METERS).sum()))

## 2. Road network

In [ ]:
if os.path.exists(GRAPHML):
    G = ox.io.load_graphml(GRAPHML)
else:
    # "bike" would be the honest network for bikes, but the point of this dataset
    # is to compare against car ETA on the same graph — keep "drive" and say so.
    G = ox.graph.graph_from_bbox(BBOX, network_type="drive", simplify=True,
                                 truncate_by_edge=True)
    ox.io.save_graphml(G, GRAPHML)
G = ox.truncate.largest_component(G, strongly=True)
print(G)

## 3. Snap the two endpoints

No route to infer, so no HMM: each endpoint just goes to its nearest edge. This is
vectorised over every ride at once, which matters at this scale.

In [ ]:
uid = assign_unique_osmids(G)
Gp = ox.projection.project_graph(G)
crs = Gp.graph["crs"]

pts = gpd.GeoSeries(gpd.points_from_xy(
    np.r_[sel.start_lng.values, sel.end_lng.values],
    np.r_[sel.start_lat.values, sel.end_lat.values]), crs="EPSG:4326").to_crs(crs)

ne, dist = ox.distance.nearest_edges(Gp, pts.x.values, pts.y.values, return_dist=True)
ne = [tuple(int(z) for z in e) for e in np.asarray(ne, dtype=object).ravel()]
n = len(sel)
start_edge, end_edge = ne[:n], ne[n:]
start_d, end_d = np.asarray(dist[:n]), np.asarray(dist[n:])
print("snap distance (m):", np.round(np.percentile(np.r_[start_d, end_d], [50, 90, 99]), 1))

ok = (start_d <= MAX_SNAP_M) & (end_d <= MAX_SNAP_M)
print(f"{ok.sum()} / {n} rides have both endpoints on the network")

In [ ]:
sel = sel[ok]
start_edge = [e for e, m in zip(start_edge, ok) if m]
end_edge = [e for e, m in zip(end_edge, ok) if m]

rows = []
for (_, r), e0, e1 in tqdm(zip(sel.iterrows(), start_edge, end_edge), total=len(sel)):
    rows.append({
        "Id": str(r["ride_id"]),
        "Coordinates": str([(round(float(r["start_lng"]), 6), round(float(r["start_lat"]), 6)),
                            (round(float(r["end_lng"]), 6), round(float(r["end_lat"]), 6))]),
        "OSMids": str([uid[e0]["unique_osmid"], uid[e1]["unique_osmid"]]),
        "Timestamps": str([int(r["t0"]), int(r["t1"])]),
        "Total_time": int(r["duration"]),
    })
print(len(rows), "OD trips")

## 4. Write the gold files

In [ ]:
matched = pd.DataFrame(rows, columns=["Id", "Coordinates", "OSMids", "Timestamps", "Total_time"])
matched.to_csv(f"{OUT}/matched_trips_{CITY}.csv", index=True)

edge_list = build_edge_list_directed(G, uid)
edge_list.to_csv(f"{OUT}/edge_list_directed_{CITY}.csv", index=False)
n_feat = build_road_network_geojson(G, uid, f"{OUT}/road_network_unique_osmids_{CITY}.geojson")

# a flag file so nobody downstream mistakes these for routes
with open(f"{OUT}/MODE_{CITY}.json", "w") as f:
    json.dump({"mode": "OD", "points_per_trip": 2, "route_observed": False,
               "note": "OSMids holds the origin and destination segment only, "
                       "not a path. Do not train a route-aware model on this file."}, f, indent=1)

print(len(matched), "trips |", len(edge_list), "transitions |", n_feat, "segments")
matched.head(2)

In [ ]:
# station-level covariates are free and useful for an OD baseline
if "start_station_id" in sel.columns:
    od = (sel.groupby(["start_station_id", "end_station_id"])
             .agg(n=("duration", "size"), median_s=("duration", "median"),
                  meters=("meters", "median"))
             .reset_index().sort_values("n", ascending=False))
    od.to_csv(f"{OUT}/od_pairs_{CITY}.csv", index=False)
    print(len(od), "distinct OD pairs; top:")
    print(od.head(3).to_string(index=False))

In [ ]:
def validate_gold(trips_path, edge_list_path=None, geojson_path=None, require_coords=True):
    """Check the produced files against the Harbin/Omsk contract."""
    df = pd.read_csv(trips_path)
    problems = []

    expected = ["Unnamed: 0", "Id", "Coordinates", "OSMids", "Timestamps", "Total_time"]
    if list(df.columns) != expected:
        problems.append(f"columns are {list(df.columns)}, expected {expected}")

    bad_len = bad_eval = bad_total = bad_coord = 0
    osmids_seen = set()
    for _, r in df.iterrows():
        try:
            c = ast.literal_eval(r["Coordinates"])
            o = ast.literal_eval(r["OSMids"])
            t = ast.literal_eval(r["Timestamps"])
        except Exception:
            bad_eval += 1
            continue
        if not (len(c) == len(o) == len(t)):
            bad_len += 1
        if t[-1] - t[0] != r["Total_time"]:
            bad_total += 1
        if require_coords and not all(isinstance(p, tuple) and len(p) == 2 for p in c):
            bad_coord += 1
        osmids_seen.update(map(str, o))

    for label, n in [("rows that do not literal_eval", bad_eval),
                     ("rows with unequal list lengths", bad_len),
                     ("rows where Total_time != Timestamps[-1] - Timestamps[0]", bad_total),
                     ("rows with malformed coordinates", bad_coord)]:
        if n:
            problems.append(f"{n} {label}")

    print(f"{trips_path}: {len(df)} trips, {len(osmids_seen)} distinct segments, "
          f"Total_time median {df['Total_time'].median():.0f} s")

    if edge_list_path:
        el = pd.read_csv(edge_list_path, dtype=str)
        if list(el.columns) != ["osmid_u", "osmid_v"]:
            problems.append(f"edge list columns are {list(el.columns)}")
        known = set(el["osmid_u"]) | set(el["osmid_v"])
        missing = osmids_seen - known
        print(f"{edge_list_path}: {len(el)} transitions, "
              f"{len(osmids_seen & known)}/{len(osmids_seen)} trip segments present")
        if missing and len(missing) > 0.05 * max(len(osmids_seen), 1):
            problems.append(f"{len(missing)} trip segments missing from the edge list")

    if geojson_path:
        with open(geojson_path) as f:
            gj = json.load(f)
        keys = {f["properties"].get("unique_osmid", f["properties"]["osmid"])
                for f in gj["features"]}
        print(f"{geojson_path}: {len(gj['features'])} features, {len(keys)} unique ids")
        if not osmids_seen <= keys:
            problems.append(f"{len(osmids_seen - keys)} trip segments missing from the geojson")

    print("\nOK — matches the gold contract" if not problems
          else "\nPROBLEMS:\n  " + "\n  ".join(problems))
    return df

In [ ]:
_ = validate_gold(f"{OUT}/matched_trips_{CITY}.csv",
                  f"{OUT}/edge_list_directed_{CITY}.csv",
                  f"{OUT}/road_network_unique_osmids_{CITY}.geojson")

## Caveats

* **Two points is not a route.** Everything above is written in the gold format for
  convenience; `MODE_nyc_citibike.json` exists so this cannot be forgotten.
* **The vehicle is a bicycle.** Not a proxy for car ETA. Route choice, rider
  fitness, e-bike vs pedal, and mid-ride stops are all folded into one label, and
  none of them are observed. The right claim is "OD benchmark with very high
  aleatoric noise", where calibration beats point accuracy.
* **The left tail is already cut** — rides under 60 s were removed by the operator
  before publication, so the label distribution is truncated, not natural.
* **Dockless e-bike coordinates are real; docked-bike coordinates are the station.**
  Mixed precision inside one column.
* **`MIN_METERS` drops round trips** (same start and end station), which are a real
  and large part of leisure riding. That is a deliberate choice — an OD pair with
  zero displacement has no distance signal at all — but it biases the sample
  towards commuting.
* Publish the notebook, not the CSVs, especially if you extend it to Divvy or
  Capital Bikeshare, whose licences forbid redistribution outright.